# Week 7: K-Nearest Neighbors & Distance Measures

## Student Practice Notebook

**Name:** ______________________
**Roll Number:** ______________________
**Date:** ______________________

## Learning Objectives

After completing this lab, you will be able to:

- Compute Euclidean and Manhattan distance between two points, by hand and in code
- Explain how KNN classifies a new point using its nearest neighbors
- Fit a `KNeighborsClassifier` correctly (split → scale on train only → fit)
- Explain why K matters, and identify overfitting/underfitting as K changes
- Solve a KNN numerical problem by hand — the kind you'll see in a written exam

### Why this week matters
Every model you've built so far (Weeks 5-6) assumed an equation. KNN assumes nothing — it just asks "what do my nearest neighbors look like?" It's the simplest possible classifier, and understanding *why* it works (and where it breaks) builds the intuition you'll need for every distance-based method later in the course.


## Part 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

%matplotlib inline
np.random.seed(1)


In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame
df['species'] = df['target'].map(dict(enumerate(iris.target_names)))
df.head()


## Part 1 — Distance Measures

KNN's entire decision rule is "find the closest points." "Closest" needs a precise definition — that's what a distance metric gives us.

### 1.1 Demo: Euclidean and Manhattan distance, by hand


In [ ]:
p1 = np.array([5.1, 3.5, 1.4, 0.2])   # a sample flower's 4 measurements
p2 = np.array([6.4, 3.2, 4.5, 1.5])   # a different flower

euclidean = np.sqrt(np.sum((p1 - p2) ** 2))
manhattan = np.sum(np.abs(p1 - p2))

print('Euclidean distance:', euclidean)
print('Manhattan distance:', manhattan)


**Predict, then think:** Euclidean distance is always ≤ or ≥ Manhattan distance for the same two points (pick one) — why? Think about what happens along a single axis vs. across multiple axes at once.

*Your answer:*


### Student Practice — Minkowski distance

The Minkowski distance generalizes both: $d(p,q) = \left(\sum |p_i - q_i|^p\right)^{1/p}$. Euclidean is Minkowski with $p=2$; Manhattan is Minkowski with $p=1$.

Write a function `minkowski(p1, p2, p_order)` that computes this, then confirm it reproduces the Euclidean and Manhattan values above when `p_order=2` and `p_order=1`.


In [ ]:
# Write your answer here


## Part 2 — Fitting a KNN Classifier

Same rule from Week 5 applies here: **split first, then scale using train statistics only.** KNN is a distance-based method, so unscaled features (e.g. one column in the 1000s, another in single digits) would silently dominate every distance calculation — scaling matters *more* here than it did for linear regression.


In [ ]:
X = df[iris.feature_names]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
knn5 = KNeighborsClassifier(n_neighbors=5)
knn5.fit(X_train_scaled, y_train)

y_pred = knn5.predict(X_test_scaled)
print('Accuracy (k=5):', accuracy_score(y_test, y_pred))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=iris.target_names).plot()
plt.show()


### Student Practice — try k=1

Fit a `KNeighborsClassifier` with `n_neighbors=1` on the same scaled data. Print its test accuracy and confusion matrix. Is it better or worse than k=5?


In [ ]:
# Write your answer here


## Part 3 — Why K Matters

### Demo: sweep K from 1 to 30


In [ ]:
train_acc, test_acc = [], []
k_values = range(1, 31)

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_acc.append(accuracy_score(y_train, knn.predict(X_train_scaled)))
    test_acc.append(accuracy_score(y_test, knn.predict(X_test_scaled)))

plt.figure(figsize=(8, 4))
plt.plot(list(k_values), train_acc, label='train accuracy')
plt.plot(list(k_values), test_acc, label='test accuracy')
plt.xlabel('K (number of neighbors)')
plt.ylabel('accuracy')
plt.legend()
plt.title('Effect of K on train vs test accuracy')
plt.show()


**Reflection:** This should look familiar from Week 6. At K=1, is the model closer to overfitting or underfitting? What about at K=30? Connect this explicitly to bias and variance.

*Your answer:*


### Student Practice — pick the best K

From the sweep above, find the K with the highest test accuracy. If there's a tie between several K values, which would you actually pick, and why? (Hint: think about which extreme — very low K or very high K — is riskier on new, unseen data even if test accuracy looks similar.)


In [ ]:
# Write your answer here


## Part 4 — Worked Numerical Problem (by hand)

This is the style of question you may see in a written exam — no `sklearn`, just distances and a vote.

**Given training data:**

| Point | Feature 1 | Feature 2 | Class |
|---|---|---|---|
| A | 2 | 4 | Red |
| B | 4 | 6 | Red |
| C | 4 | 2 | Blue |
| D | 6 | 4 | Blue |
| E | 6 | 2 | Blue |

**Query point Q = (5, 5). Using K=3 and Euclidean distance, what class does Q belong to?**

### Student Practice — solve it by hand first, then verify in code

1. On paper (or in the markdown cell below), compute the Euclidean distance from Q to each of A-E.
2. List the 3 nearest points and their classes.
3. Take the majority vote — what's your predicted class for Q?
4. THEN run the code cell below to check your work.

*Your hand-worked answer:*


In [ ]:
points = {'A': (2, 4, 'Red'), 'B': (4, 6, 'Red'), 'C': (4, 2, 'Blue'), 'D': (6, 4, 'Blue'), 'E': (6, 2, 'Blue')}
Q = np.array([5, 5])

distances = {}
for name, (x, y_, label) in points.items():
    d = np.sqrt((Q[0] - x) ** 2 + (Q[1] - y_) ** 2)
    distances[name] = (round(d, 3), label)

for name, (d, label) in sorted(distances.items(), key=lambda item: item[1][0]):
    print(f'{name}: distance={d}, class={label}')


**Reflection:** Did your hand-computed answer match the code's ranking? If not, find where the arithmetic went wrong — this is exactly the kind of check-your-work habit that pays off in a written exam.

*Your answer:*


# Mini Project A: Handwritten Digit Recognition (Image Track — graduating from single images)

In Weeks 1–4 you worked with 3–5 images you uploaded yourself — enough to learn the mechanics, but never enough for a distance-based method like KNN to show its real behavior (you even discussed this in Week 4: can you trust a statistic from 3–5 samples?). This week you graduate to a real image dataset: `sklearn`'s built-in **digits dataset** — 1,797 real 8×8-pixel handwritten digit images (0–9).

### Demo: load and view a digit as both an image and a feature vector


In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print('Images shape:', digits.images.shape)   # (1797, 8, 8) -- 1797 images, each 8x8 pixels
print('Data shape:  ', digits.data.shape)     # (1797, 64)   -- each image FLATTENED into 64 features

plt.figure(figsize=(3, 3))
plt.imshow(digits.images[0], cmap='gray')
plt.title(f'Label: {digits.target[0]}')
plt.show()


Notice: `digits.data` is just `digits.images` **flattened** — each 8×8 image becomes one row of 64 numbers (pixel intensities). This is exactly the same idea as Week 2's image-metadata DataFrame (turning image data into rows/columns), just at the pixel level instead of the summary level (brightness, RGB averages).

### Student Practice — build and evaluate a digit classifier

1. Split `digits.data`/`digits.target` into train/test (`stratify=digits.target`, same as Part 2).
2. Scale with `StandardScaler` — fit on train only, same discipline as before.
3. Fit a `KNeighborsClassifier` with a K of your choice.
4. Print test accuracy and show the confusion matrix.
5. Find and display (with `plt.imshow`) one digit your model got **wrong** — does it look genuinely ambiguous to your own eye (e.g. a messy 4 that looks like a 9)?


In [ ]:
# Write your answer here


**Reflection:** With 1,797 images instead of 3–5, does K=1 vs K=5 vs K=15 now show a clearer, more trustworthy pattern than it would have on your tiny self-uploaded set? Why does sample size matter here in exactly the same way it mattered for the t-test in Week 4?

*Your answer:*


# Mini Project B: Classifying Text by Topic with KNN (NLP Track — building on Week 1's vectors)

In Week 1 you built bag-of-words vectors from scratch with NumPy and compared sentences using cosine similarity. KNN classification is the natural next step: if two documents' word-vectors are *close*, they're probably about the same topic.

### Demo: bag-of-words vectors for short labeled sentences


In [ ]:
sentences = [
    'the cricket team won the match',
    'the batsman scored a century today',
    'chop the onions and boil the pasta',
    'add salt and simmer the curry',
    'the new phone has a faster processor',
    'the laptop battery lasts all day',
]
labels = ['sports', 'sports', 'cooking', 'cooking', 'tech', 'tech']

# Build vocabulary (same idea as Week 1's bag-of-words, done manually)
vocab = sorted(set(word for s in sentences for word in s.lower().split()))
print('Vocabulary size:', len(vocab))

def to_vector(sentence, vocab):
    words = sentence.lower().split()
    return np.array([words.count(w) for w in vocab])

X_text = np.array([to_vector(s, vocab) for s in sentences])
X_text.shape


### Student Practice — classify a new sentence

1. Fit a `KNeighborsClassifier` (try K=1 or K=3, since we only have 6 training sentences — discuss why a large K wouldn't make sense here) on `X_text`/`labels`.
2. Convert a **new sentence you write yourself** (e.g. `'the striker scored a goal'`) into a vector using the same `to_vector(sentence, vocab)` function.
3. Predict its topic. Does it match what you'd expect?
4. Try a deliberately ambiguous sentence that mixes two topics — what does the model predict, and does that expose a limitation of bag-of-words (it only counts words, it doesn't understand meaning)?


In [ ]:
# Write your answer here


**Reflection:** With only 6 training sentences, why would K=5 or K=6 be a bad choice here, no matter how well it scored on this tiny set? Connect this back to the bias-variance discussion from Part 3 — this is the same principle showing up in a completely different kind of data (text instead of numbers/pixels).

*Your answer:*


## Summary

| What we did | Why it matters |
|---|---|
| Euclidean / Manhattan / Minkowski distance | The mechanism every distance-based method (KNN, clustering) relies on |
| KNN classifier, split-then-scale | Same leakage discipline as Week 5, now applied to a classifier |
| K sweep | Low K = overfitting (high variance); high K = underfitting (high bias) — same shape as Week 6's bias-variance curve, different knob |
| Hand-worked numerical problem | Exam-style practice — you should be able to do this without code |
| Mini Project A (Image) | KNN on real, large-scale image data (digits) — finally enough samples to trust the pattern |
| Mini Project B (NLP) | KNN on text vectors — same algorithm, completely different kind of data |

**Next week:** Decision Trees (ID3) — a completely different way of drawing a decision boundary, and your first look at feature-based (rather than distance-based) classification.
